####  [ 사전 학습 - AlexNet Model 기반 Cat & Dog 분류 ] 


In [ ]:
## 모듈 로딩
import torch
import torch.nn as nn   
import torch.nn.functional as F

import torchvision.transforms as transforms
from torchvision.models import *    ## 사전학습 내장 모델 클래스 관련
from torchinfo import summary       ## 모델 정보 확인용

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [2]:
# 데이터 준비
DATA_DIR = '../_data/image/cat_dog/'

## 이미지 전처리 준비
PREPROCESSING = AlexNet_Weights.DEFAULT.transforms()
# CLASSES = AlexNet_Weights.DEFAULT.meta['categories']
# IDX_TO_CLASS = dict(zip(range(1000), CLASSES))

[2] 이미지 데이터 로딩

In [3]:
## - 이미지 데이터 로딩
imgDS = ImageFolder(root=DATA_DIR,
                    transform=PREPROCESSING)

IDX_TO_CLASS = {v:k for k, v in imgDS.class_to_idx.items()}
print(f'IDX_TO_CLASS => {IDX_TO_CLASS}')

IDX_TO_CLASS => {0: 'cat', 1: 'dog'}


In [4]:
## - 데이터 확인
print(f'imgDataset 개수 : {len(imgDS.targets)}')
print(f'imgDataset 분류 : {imgDS.class_to_idx}')
print(f'cat 개수 : {(imgDS.targets.count(0))}, {(imgDS.targets.count(0)/len(imgDS.targets))*100}')
print(f'dog 개수 : {(imgDS.targets.count(1))}, {(imgDS.targets.count(1)/len(imgDS.targets))*100}')

imgDataset 개수 : 120
imgDataset 분류 : {'cat': 0, 'dog': 1}
cat 개수 : 60, 50.0
dog 개수 : 60, 50.0


In [5]:
# 데이터 확인
img, label = imgDS[0]
print(f'img {img.shape}, label {label}')

img torch.Size([3, 224, 224]), label 0


[3] 모델 클래스 정의 및 선언<hr>

In [6]:
## 모델 구성: 사전 학습된 AlexNet의 특징추출부분 + 커스텀 분류기 부분
""" 
특징추출 부분 : 최적의 W, b 설정 => 학습 X  => requires_grad = False 설정
분류기 부분 : 커스터마이징 => 학습 필요     => requires_grad = True 설정

"""

' \n특징추출 부분 : 최적의 W, b 설정 => 학습 X  => requires_grad = False 설정\n분류기 부분 : 커스터마이징 => 학습 필요     => requires_grad = True 설정\n\n'

In [7]:
# (1) 사전 학습된 모델 인스턴스 로딩
model = alexnet(weights= AlexNet_Weights.DEFAULT)

In [8]:
## 모델 층별 특성 확인
## 모델 층별 W,b 파라미터 업데이트 설정
## ==> model.feature 부분 : requires_grad = False
## ==> model.classifier 부분 : 2진 분류기로 변경
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)
    if name.startswith('features'): 
        param.requires_grad=False
        
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)     
       

features.0.weight torch.Size([64, 3, 11, 11]) True
features.0.bias torch.Size([64]) True
features.3.weight torch.Size([192, 64, 5, 5]) True
features.3.bias torch.Size([192]) True
features.6.weight torch.Size([384, 192, 3, 3]) True
features.6.bias torch.Size([384]) True
features.8.weight torch.Size([256, 384, 3, 3]) True
features.8.bias torch.Size([256]) True
features.10.weight torch.Size([256, 256, 3, 3]) True
features.10.bias torch.Size([256]) True
classifier.1.weight torch.Size([4096, 9216]) True
classifier.1.bias torch.Size([4096]) True
classifier.4.weight torch.Size([4096, 4096]) True
classifier.4.bias torch.Size([4096]) True
classifier.6.weight torch.Size([1000, 4096]) True
classifier.6.bias torch.Size([1000]) True
features.0.weight torch.Size([64, 3, 11, 11]) False
features.0.bias torch.Size([64]) False
features.3.weight torch.Size([192, 64, 5, 5]) False
features.3.bias torch.Size([192]) False
features.6.weight torch.Size([384, 192, 3, 3]) False
features.6.bias torch.Size([384]) 

In [9]:
## (2) 분류기 부분 변경 => 입력 (BS, 9216) => 출력 (9216 , 2)
## 
model.classifier =nn.Sequential(
                nn.Linear(9216, 4096),
                nn.ReLU(),
                nn.Linear(4096,1)
                
    ) 
summary(model, input_size=(1,3,224,224) )
 

Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [1, 1]                    --
├─Sequential: 1-1                        [1, 256, 6, 6]            --
│    └─Conv2d: 2-1                       [1, 64, 55, 55]           (23,296)
│    └─ReLU: 2-2                         [1, 64, 55, 55]           --
│    └─MaxPool2d: 2-3                    [1, 64, 27, 27]           --
│    └─Conv2d: 2-4                       [1, 192, 27, 27]          (307,392)
│    └─ReLU: 2-5                         [1, 192, 27, 27]          --
│    └─MaxPool2d: 2-6                    [1, 192, 13, 13]          --
│    └─Conv2d: 2-7                       [1, 384, 13, 13]          (663,936)
│    └─ReLU: 2-8                         [1, 384, 13, 13]          --
│    └─Conv2d: 2-9                       [1, 256, 13, 13]          (884,992)
│    └─ReLU: 2-10                        [1, 256, 13, 13]          --
│    └─Conv2d: 2-11                      [1, 256, 13, 13] 

In [ ]:
from torch.utils.data import Subset
import numpy afasdasdadss np

In [11]:
num_samples = len(imgDS)
indices = np.arange(num_samples)
np.random.shuffle(indices)

In [12]:
trainDS = Subset(imgDS, indices=indices[:int(num_samples*0.8)])
valDS = Subset(imgDS, indices=indices[int(num_samples*0.8):])


In [13]:
val_len = len(valDS)
val_indicies = np.arange(val_len)

testDS = Subset(valDS, indices=val_indicies[:int(val_len*0.5)])
valDS = Subset(valDS, indices=val_indicies[int(val_len*0.5):])

In [14]:
len(trainDS)

96

In [15]:
trainDL = DataLoader(trainDS, batch_size=48, shuffle=True, num_workers=4)
valDL = DataLoader(valDS, shuffle=True, num_workers=4)
testDL = DataLoader(testDS, shuffle=True, num_workers=4)

[4] 학습준비<hr>

In [16]:
import torch.optim as optim 
from torch.optim.lr_scheduler import StepLR

In [17]:
## 학습 관련 설정 값

INPUT_SIZE = (1,3,224,224)
OUPUT_SIZE = 1

LR              = 0.01
EPOCHS          = 100
STEP_SIZE       = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
## 학습 관련 인스턴스들 생성
MODEL = model
MODEL.to(DEVICE)

LOSS_FN   = nn.BCEWithLogitsLoss()
OPTIMIZER = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

SCHEDULER = StepLR(OPTIMIZER, STEP_SIZE, gamma=0.1)

In [19]:
# for name in MODEL.parameters():
#     print( name)

In [20]:
## 학습 관련 함수들

In [21]:
## -------------------------------------------------------------------------
## 함수기능 : 학습데이터를 사용하여 학습 진행
## 함수이름 : training
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def training(dataloader):
    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text.float())

        loss = LOSS_FN(pre, label.reshape(-1,1).float())
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1



In [22]:
## -------------------------------------------------------------------------
## 함수기능 : 검증데이터를 사용하여 학습 진행
## 함수이름 : evaluate
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def evaluate(dataloader):
    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text.float())
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1,1).float())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

[5] 학습 진행 <hr>

In [23]:
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(trainDL)
    valid_loss, valid_acc = evaluate(valDL)
    SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print(f'| end of epoch {epoch:3d} | train loss {train_loss:8.3f}  | valid loss {valid_loss:8.3f}')
    print("-" * 59)

    ## 모델 저장 
    if MAX_ACC < valid_acc : 
        # torch.save(MODEL, MODEL_DIR+MODEL_FILE)
        MAX_ACC = valid_acc


-----------------------------------------------------------
| end of epoch   1 | train acc   52.000  | valid acc    1.200
| end of epoch   1 | train loss   77.032  | valid loss   34.537
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   2 | train acc   52.000  | valid acc    1.400
| end of epoch   2 | train loss   28.257  | valid loss    1.000
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   3 | train acc   52.000  | valid acc    1.400
| end of epoch   3 | train loss    1.000  | valid loss    1.000
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   4 | train acc   52.000  | valid acc    1.600
| end of epoch   4 | train loss    1.000  | valid loss    1.000
-----------------------------------------------------------
----------------

KeyboardInterrupt: 

In [24]:
## 이미지 데이터 로딩 후 전처리 => 모델 => 예측값 체크

from PIL import Image
import matplotlib.pyplot as plt

def check(imgfile):    
    img = Image.open(imgfile)
    print(img)
    img_transe = PREPROCESSING(img)
    print(img_transe.shape)
    
    img_transe.unsqueeze_(0)
    pre = model(img_transe)
    print(pre.argmax())
    print(pre.max(dim=1)[1].item())
    print(IDX_TO_CLASS[pre.argmax().item()])
    plt.imshow(img)
    plt.title(imgfile.split('/')[3][:-4])
    plt.show()




In [25]:
import os
catlist = os.listdir(DATA_DIR+'cat/')
catlist = [DATA_DIR+'cat/'+x for x in catlist]
doglist = os.listdir(DATA_DIR+'dog/')
doglist = [DATA_DIR+'dog/'+x for x in doglist]


In [26]:
count = 0 
for imgfile in catlist:
    img = Image.open(imgfile)
    img_transe = PREPROCESSING(img)
    img_transe.unsqueeze_(0)
    pre = model(img_transe)
    # print(pre.argmax())
    # print(pre.max(dim=1)[1].item())
    # print(IDX_TO_CLASS[pre.argmax().item()])
    # if imgfile.split('/')[5] == pre:
    if pre.argmax() == 0:
        count += 1
print(count)

60


In [27]:
count = 0 
for imgfile in doglist:
    img = Image.open(imgfile)
    img_transe = PREPROCESSING(img)
    img_transe.unsqueeze_(0)
    pre = model(img_transe)
    # print(pre.argmax())
    # print(pre.max(dim=1)[1].item())
    # print(IDX_TO_CLASS[pre.argmax().item()])
    # if imgfile.split('/')[5] == pre:
    if pre.argmax() == 0:
        count += 1
print(count)

60


In [33]:
len(testDL)

12

In [32]:
count = 0 
for imgfile, label in testDL:
    pre = model(imgfile)
    if pre.argmax() == label:
        count += 1
print(count)

6
